In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppresses TensorFlow warnings

# Optional: suppress logging from TensorFlow itself
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

E0000 00:00:1748940570.124464      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748940570.187469      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
# === Install Requirements (if not already installed) ===
!pip install typeguard==4.0.1
!pip install -q tensorflow tensorflow-addons

# === Imports ===
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import image_dataset_from_directory

# === Constants ===
IMG_SIZE = (160, 160)
BATCH_SIZE = 16
EPOCHS = 10
SEED = 42

# === Directories ===
base_dir = "/kaggle/input/eggplant/Eggplant "
train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "val")
test_dir = os.path.join(base_dir, "test")

# === Load Datasets ===
train_dataset = image_dataset_from_directory(
    train_dir,
    shuffle=True,
    labels='inferred',
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    seed=SEED
)

validation_dataset = image_dataset_from_directory(
    val_dir,
    shuffle=True,
    labels='inferred',
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    seed=SEED
)

test_dataset = image_dataset_from_directory(
    test_dir,
    shuffle=False,
    labels='inferred',
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    seed=SEED
)

# === Class Names ===
class_names = train_dataset.class_names
num_classes = len(class_names)

# === Class Weights Calculation ===
data_imbalance = [len(glob.glob(os.path.join(train_dir, class_folder, '*.*'))) for class_folder in class_names]
total_images = sum(data_imbalance)
class_weight = {
    i: (1.0 / count) * (total_images / len(data_imbalance))
    for i, count in enumerate(data_imbalance)
}

# === Visualize Class Distribution ===
plt.figure(figsize=(13, 6))
sns.barplot(x=class_names, y=data_imbalance, palette="rocket")
plt.title("Class Distribution")
plt.ylabel("Number of Images")
plt.xlabel("Disease Class")
plt.xticks(rotation=15)
plt.show()

# === Data Augmentation & Preprocessing ===
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
])

# === Visualize Augmented Images ===
for image, _ in train_dataset.take(1):
    plt.figure(figsize=(10, 10))
    first_image = image[0]
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        augmented_image = data_augmentation(tf.expand_dims(first_image, 0))
        img = tf.cast(augmented_image[0], tf.float32) / 255.0
        plt.imshow(img)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

# === Dataset Mapping ===
def augment_image(image, label):
    image = data_augmentation(image)
    return image, label

def preprocess_image(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

# === Dataset Mapping ===
train_dataset = train_dataset.map(augment_image).map(preprocess_image).prefetch(tf.data.AUTOTUNE)
validation_dataset = validation_dataset.map(preprocess_image).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.map(preprocess_image).prefetch(tf.data.AUTOTUNE)


# === CvT Helper Functions ===
def conv_embedding(x, filters, kernel_size=3, strides=1, name="conv_embed"):
    x = layers.DepthwiseConv2D(kernel_size, strides=strides, padding='same', name=f"{name}_depthwise")(x)
    x = layers.Conv2D(filters, 1, padding='same', name=f"{name}_pointwise")(x)
    x = layers.BatchNormalization(name=f"{name}_bn")(x)
    x = layers.Activation("gelu", name=f"{name}_gelu")(x)
    return x

def transformer_encoder(x, num_heads, projection_dim, dropout=0.1, name="transformer"):
    H, W, C = x.shape[1], x.shape[2], x.shape[3]
    x_flat = layers.Reshape((-1, C))(x)

    x1 = layers.LayerNormalization(epsilon=1e-6, name=f"{name}_ln1")(x_flat)
    attn_output = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=projection_dim, dropout=dropout, name=f"{name}_mha"
    )(x1, x1)
    x2 = layers.Add(name=f"{name}_skip1")([x_flat, attn_output])

    x3 = layers.LayerNormalization(epsilon=1e-6, name=f"{name}_ln2")(x2)
    ffn_output = keras.Sequential([
        layers.Dense(projection_dim * 4, activation="gelu"),
        layers.Dense(projection_dim),
    ], name=f"{name}_ffn")(x3)
    x4 = layers.Add(name=f"{name}_skip2")([x2, ffn_output])

    x_out = layers.Reshape((H, W, C))(x4)
    return x_out

def transformer_block_stage(x, num_blocks, num_heads, projection_dim, name="stage"):
    for i in range(num_blocks):
        x = transformer_encoder(x, num_heads=num_heads, projection_dim=projection_dim, name=f"{name}_block{i}")
    return x

# === Create CvT Model ===
def create_cvt_model(input_shape=(160, 160, 3), num_classes=10):
    inputs = keras.Input(shape=input_shape)

    x = conv_embedding(inputs, filters=64, strides=2, name="stage1_embed")
    x = transformer_block_stage(x, num_blocks=2, num_heads=1, projection_dim=64, name="stage1")

    x = conv_embedding(x, filters=128, strides=2, name="stage2_embed")
    x = transformer_block_stage(x, num_blocks=2, num_heads=2, projection_dim=128, name="stage2")

    x = conv_embedding(x, filters=256, strides=2, name="stage3_embed")
    x = transformer_block_stage(x, num_blocks=2, num_heads=4, projection_dim=256, name="stage3")

    x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="classifier")(x)

    return keras.Model(inputs, outputs, name="CvT_Enhanced")

# === Compile and Train Model ===
model = create_cvt_model(input_shape=(160, 160, 3), num_classes=num_classes)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(patience=2)
    ]
)

model.save('/kaggle/working/rice_disease_cvt_model.h5')

  Attempting uninstall: typeguard
    Found existing installation: typeguard 4.4.2
    Uninstalling typeguard-4.4.2:
      Successfully uninstalled typeguard-4.4.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.8/611.8 kB 21.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.16.1 requires typeguard<5,>=3, but you have typeguard 2.13.3 which is incompatible.
inflect 7.5.0 requires typeguard>=4.0.1, but you have typeguard 2.13.3 which is incompatible.


NotFoundError: Could not find directory /kaggle/input/eggplant/Eggplant /train

In [ ]:
test_loss, test_acc = model.evaluate(validation_dataset)
print(f"Validation Accuracy: {test_acc:.2f}")


In [ ]:
def plot_training(history):
    acc = history.history["accuracy"]
    val_acc = history.history["val_accuracy"]
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]
    epochs_range = range(len(acc))

    plt.figure(figsize=(14, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label="Training Accuracy")
    plt.plot(epochs_range, val_acc, label="Validation Accuracy")
    plt.legend(loc="lower right")
    plt.title("Training and Validation Accuracy")

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label="Training Loss")
    plt.plot(epochs_range, val_loss, label="Validation Loss")
    plt.legend(loc="upper right")
    plt.title("Training and Validation Loss")
    plt.show()

plot_training(history)


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns


# Get predictions
y_true = []
y_pred = []

for images, labels in validation_dataset:
    preds = model.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=class_names, yticklabels=class_names, cmap="YlGnBu")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.xticks(rotation=45)
plt.yticks(rotation=45)
plt.tight_layout()
plt.show()

# Classification report
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))
